## 텍스트 유사도와 추천 시스템 (Text Similarity)

### 0. 텍스트 유사도 (Text Similarity)

* 지금까지: 텍스트 $\rightarrow$ 토큰화 $\rightarrow$ 정수 인코딩 $\rightarrow$ 임베딩 $\rightarrow$ LSTM 으로 리뷰가 긍정인지 부정인지 **분류**
* 이번 시간: 두 문서(영화 줄거리)가 **얼마나 비슷한지**를 숫자로 계산 $\rightarrow$ 줄거리가 비슷한 영화를 **추천**
* 딥러닝 없이 "단어가 몇 번 나왔는지 세는 것"만으로도 가능
    - 문서를 숫자 벡터로 바꾸기: Bag of Words, TF-IDF
    - 두 벡터가 얼마나 비슷한지 재기: 코사인 유사도

In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter

### 1. Bag of Words (BoW)

#### 1) BoW란?

* 단어의 **순서는 무시**하고, 각 단어가 **몇 번 등장했는지(빈도)** 만 세어서 문서를 숫자 벡터로 표현하는 방법
    - 단어들을 가방(bag)에 몽땅 넣고 흔들어 섞은 뒤, 종류별로 몇 개씩 있는지 세는 것과 같음
* 만드는 과정
    1. 각 단어에 고유한 정수 인덱스를 부여 (= 단어 집합 만들기, 10-2에서 한 것과 동일)
    2. 각 인덱스 위치에 해당 단어의 등장 횟수를 기록한 벡터를 만듦

In [2]:
doc = '먹고 싶은 사과 먹고 싶은 바나나 바나나'
tokens = doc.split() # 띄어쓰기 기준으로 토큰화
print(tokens)

['먹고', '싶은', '사과', '먹고', '싶은', '바나나', '바나나']


In [3]:
word_to_index = {}
bow = []

for word in tokens:
    if word not in word_to_index:
        word_to_index[word] = len(word_to_index) # 처음 본 단어: 새 인덱스 부여
        bow.append(1)                            # 등장 횟수 1로 시작
    else:
        bow[word_to_index[word]] += 1            # 이미 본 단어: 등장 횟수 +1

print('단어 집합 :', word_to_index)
print('BoW 벡터 :', bow)

단어 집합 : {'먹고': 0, '싶은': 1, '사과': 2, '바나나': 3}
BoW 벡터 : [2, 2, 1, 2]


* `먹고`, `싶은`은 2번, `사과`는 1번, `바나나`는 2번 등장 $\rightarrow$ `[2, 2, 1, 2]`
* "바나나 바나나 먹고 싶은 사과 먹고 싶은" 처럼 순서를 바꿔도 BoW 벡터는 같음

#### 2) 여러 문서를 같은 기준으로 표현하기

* 문서마다 단어 집합을 따로 만들면 벡터의 길이와 인덱스가 달라져서 서로 비교할 수 없음
* **모든 문서의 단어**로 단어 집합을 하나 만들고, 각 문서를 그 단어 집합 기준으로 BoW 벡터로 바꿈

In [4]:
def build_vocab(docs):
    """토큰화된 문서들(docs)에 등장하는 모든 단어로 단어 집합을 만듦"""
    words = set()
    for tokens in docs:
        words.update(tokens)                    # 중복 없이 단어를 모음
    return {word: index for index, word in enumerate(sorted(words))}  # 가나다순으로 인덱스 부여

def make_bow(tokens, word_to_index):
    """토큰 리스트를 단어 집합 기준의 BoW 벡터로 바꿈"""
    bow = [0] * len(word_to_index)              # 단어 집합 크기만큼 0으로 채움
    for word in tokens:
        bow[word_to_index[word]] += 1
    return bow

In [5]:
docs = [
    '먹고 싶은 사과'.split(),
    '먹고 싶은 바나나'.split(),
]
word_to_index = build_vocab(docs)
print('단어 집합 :', word_to_index)
print('문서1 BoW :', make_bow(docs[0], word_to_index))
print('문서2 BoW :', make_bow(docs[1], word_to_index))

단어 집합 : {'먹고': 0, '바나나': 1, '사과': 2, '싶은': 3}
문서1 BoW : [1, 0, 1, 1]
문서2 BoW : [1, 1, 0, 1]


* 두 벡터의 길이가 같고(5), 같은 위치가 같은 단어를 뜻하므로 서로 비교할 수 있음
* `사과`, `바나나` 자리만 다르고 `먹고`, `싶은` 자리는 같음 $\rightarrow$ "두 문서가 비슷하다"를 숫자로 표현할 준비가 됨

#### 3) 불용어를 제거한 BoW

* 불용어(stopwords): 자주 등장하지만 의미 분석에는 거의 기여하지 않는 단어 (영어: the, a, is ...)
* BoW는 빈도를 세는 방법이므로, 불용어를 빼지 않으면 의미 없는 단어가 가장 큰 값을 갖게 됨
* 영어 문장은 10-2에서 배운 정규 표현식으로 간단히 토큰화할 수 있음

In [6]:
text = "Family is not an important thing. It's everything."

tokens = re.findall('[a-z]+', text.lower()) # 소문자로 바꾼 뒤 알파벳이 연속된 덩어리만 추출
print(tokens)

['family', 'is', 'not', 'an', 'important', 'thing', 'it', 's', 'everything']


In [7]:
stopwords = ['is', 'not', 'an', 'it', 's']
tokens = [word for word in tokens if word not in stopwords] # 불용어 제거 (10-2와 같은 방법)
print(tokens)

word_to_index = build_vocab([tokens])
print('단어 집합 :', word_to_index)
print('BoW 벡터 :', make_bow(tokens, word_to_index))

['family', 'important', 'thing', 'everything']
단어 집합 : {'everything': 0, 'family': 1, 'important': 2, 'thing': 3}
BoW 벡터 : [1, 1, 1, 1]


### 2. 문서 단어 행렬(DTM)과 TF-IDF

#### 1) 문서 단어 행렬 (Document-Term Matrix, DTM)

* 여러 문서의 BoW 벡터를 행으로 쌓은 행렬
    - 행: 문서, 열: 단어, 값: 해당 문서에서 해당 단어의 등장 횟수
* 문서들을 같은 기준(같은 단어 집합)으로 표현하므로 문서끼리 비교할 수 있게 됨

In [8]:
corpus = [
    '먹고 싶은 사과',
    '먹고 싶은 바나나',
    '길고 노란 바나나 바나나',
    '저는 과일이 좋아요',
]

docs = [doc.split() for doc in corpus]                      # 토큰화
word_to_index = build_vocab(docs)                            # 단어 집합
dtm = np.array([make_bow(tokens, word_to_index) for tokens in docs]) # 각 문서의 BoW를 행으로 쌓음

# 보기 좋게 데이터프레임으로 출력
pd.DataFrame(dtm, columns=word_to_index.keys(), index=['문서1', '문서2', '문서3', '문서4'])

,과일이,길고,노란,먹고,바나나,사과,싶은,저는,좋아요
문서1,0,0,0,1,0,1,1,0,0
문서2,0,0,0,1,1,0,1,0,0
문서3,0,1,1,0,2,0,0,0,0
문서4,1,0,0,0,0,0,0,1,1


**DTM의 한계**

* 희소 표현(sparse representation): 단어 집합이 커질수록 대부분의 값이 0인 거대한 행렬이 됨
* 단순 빈도 기반: 모든 문서에 자주 등장하는 단어(영어의 the, 영화 리뷰의 "영화")가 마치 중요한 단어인 것처럼 큰 값을 가짐
    - 문서1과 문서2는 `먹고`, `싶은` 때문에 비슷해 보이지만, 정작 중요한 차이는 `사과`와 `바나나`

#### 2) TF-IDF (Term Frequency-Inverse Document Frequency)

* 아이디어: **이 문서에는 많이 나오고(TF), 다른 문서에는 잘 안 나오는(IDF) 단어**가 그 문서를 잘 설명하는 단어
* $tf(d, t)$: 문서 $d$에서 단어 $t$의 등장 횟수 (= DTM의 값)
* $df(t)$: 단어 $t$가 등장한 **문서의 수**
* $idf(t) = \log \left( \frac{N}{1 + df(t)} \right)$, $N$: 전체 문서 수
    - 많은 문서에 등장하는 단어일수록 $idf$가 작아짐 $\rightarrow$ 중요도가 낮아짐
    - 분모의 $1$은 0으로 나누는 것을 막기 위한 것
* $\text{TF-IDF}(d, t) = tf(d, t) \times idf(t)$

In [9]:
N = len(docs)                     # 전체 문서 수
df = (dtm > 0).sum(axis=0)        # 각 단어가 등장한 문서 수 (열 방향으로 0이 아닌 값의 개수)
idf = np.log(N / (1 + df))

pd.DataFrame({'df': df, 'idf': idf.round(2)}, index=word_to_index.keys()).T

,과일이,길고,노란,먹고,바나나,사과,싶은,저는,좋아요
df,1.00,1.00,1.00,2.00,2.00,1.00,2.00,1.00,1.00
idf,0.69,0.69,0.69,0.29,0.29,0.69,0.29,0.69,0.69


In [10]:
tfidf = dtm * idf                 # 각 열(단어)에 해당 단어의 idf를 곱함

pd.DataFrame(tfidf, columns=word_to_index.keys(), index=['문서1', '문서2', '문서3', '문서4']).round(2)

,과일이,길고,노란,먹고,바나나,사과,싶은,저는,좋아요
문서1,0.00,0.00,0.00,0.29,0.00,0.69,0.29,0.00,0.00
문서2,0.00,0.00,0.00,0.29,0.29,0.00,0.29,0.00,0.00
문서3,0.00,0.69,0.69,0.00,0.58,0.00,0.00,0.00,0.00
문서4,0.69,0.00,0.00,0.00,0.00,0.00,0.00,0.69,0.69


* 문서1에서 두 문서에 걸쳐 등장하는 `먹고`, `싶은`(0.29)보다 문서1에만 등장하는 `사과`(0.69)의 값이 더 큼
* 문서3의 `바나나`는 2번 등장해서 0.58이지만, 두 문서에 등장하므로 한 문서에만 나오는 `길고`, `노란`(0.69)보다는 작음
* 실제 라이브러리들은 $idf$가 음수가 되지 않도록 식을 조금 바꿔서 쓰지만, 기본 아이디어는 같음

### 3. 코사인 유사도 (Cosine Similarity)

* 두 벡터 사이의 **각도**로 유사도를 재는 방법
    - 방향이 완전히 같으면 1, 90°면 0, 반대 방향이면 -1
    - BoW, TF-IDF 벡터는 음수가 없으므로 0 ~ 1 사이의 값을 가짐. 1에 가까울수록 비슷한 문서

$$
similarity = \cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}
$$

In [11]:
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

In [12]:
# 문서1: 저는 사과 좋아요
# 문서2: 저는 바나나 좋아요
# 문서3: 저는 바나나 좋아요 저는 바나나 좋아요
# 단어 순서: [바나나, 사과, 저는, 좋아요]
doc1 = np.array([0, 1, 1, 1])
doc2 = np.array([1, 0, 1, 1])
doc3 = np.array([2, 0, 2, 2])

print('문서1 vs 문서2 :', cos_sim(doc1, doc2))
print('문서1 vs 문서3 :', cos_sim(doc1, doc3))
print('문서2 vs 문서3 :', cos_sim(doc2, doc3))

문서1 vs 문서2 : 0.6666666666666667
문서1 vs 문서3 : 0.6666666666666667
문서2 vs 문서3 : 1.0000000000000002


* 문서3은 문서2를 두 번 반복한 것 $\rightarrow$ 벡터의 크기는 다르지만 방향이 같아서 유사도가 1
* 코사인 유사도는 문서의 **길이에 영향을 덜 받음** (긴 문서라고 해서 유리하지 않음)

**모든 문서 쌍의 유사도를 한 번에 구하기**

* 문서가 많아지면 `cos_sim`을 쌍마다 호출하는 대신 행렬 연산으로 한 번에 계산
    1. 각 문서 벡터를 자기 길이로 나누어 길이를 1로 맞춤 ($\|A\| = 1$)
    2. 길이가 1인 벡터끼리의 내적은 곧 코사인 유사도 $\rightarrow$ 행렬 곱 한 번으로 모든 쌍을 계산

In [13]:
def cos_sim_matrix(X):
    """X의 각 행(문서 벡터)끼리의 코사인 유사도 행렬을 구함"""
    norms = np.linalg.norm(X, axis=1, keepdims=True) # 각 문서 벡터의 길이
    norms[norms == 0] = 1                            # 길이가 0인 빈 문서는 0으로 나누지 않도록
    X_normalized = X / norms                         # 길이를 1로 맞춤
    return X_normalized @ X_normalized.T             # 행렬 곱 → (문서 수, 문서 수)

In [14]:
# 2절에서 만든 TF-IDF 행렬로 네 문서 사이의 유사도 계산
sim = cos_sim_matrix(tfidf)
pd.DataFrame(sim, index=['문서1', '문서2', '문서3', '문서4'], columns=['문서1', '문서2', '문서3', '문서4']).round(2)

,문서1,문서2,문서3,문서4
문서1,1.00,0.41,0.00,0.0
문서2,0.41,1.00,0.29,0.0
문서3,0.00,0.29,1.00,0.0
문서4,0.00,0.00,0.00,1.0


* 대각선은 자기 자신과의 유사도이므로 항상 1
* 문서1(사과)과 문서2(바나나)는 `먹고 싶은` 때문에 0.41, 문서2와 문서3은 `바나나` 때문에 0.29
* 문서4(과일이 좋아요)는 다른 문서와 겹치는 단어가 없어서 모두 0

### 4. 줄거리 기반 영화 추천 시스템 만들기

* 좋아하는 영화를 입력하면 **줄거리가 비슷한 영화**를 추천해 주는 시스템
* 영화 줄거리 $\rightarrow$ TF-IDF 벡터 $\rightarrow$ 모든 영화 쌍의 코사인 유사도 $\rightarrow$ 유사도가 높은 순으로 추천
* 데이터: TMDB 5000 영화 데이터 (영화 4,803편의 제목, 줄거리, 장르, 감독 등 24개 열)

#### 1) 데이터 로드

In [15]:
import os
import urllib.request

os.makedirs('./Data/movies', exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/codeheroku/Introduction-to-Machine-Learning/master/Building%20a%20Movie%20Recommendation%20Engine/movie_dataset.csv', filename='./Data/movies/movie_dataset.csv') # 약 23MB

('./Data/movies/movie_dataset.csv', <http.client.HTTPMessage at 0x2340ff80dc0>)

In [16]:
data = pd.read_csv('./Data/movies/movie_dataset.csv')
print('영화 개수 :', len(data))
data.columns

영화 개수 : 4803


Index(['index', 'budget', 'genres', 'homepage', 'id', 'keywords',
       'original_language', 'original_title', 'overview', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'cast', 'crew', 'director'],
      dtype='object')

In [17]:
# 사용할 열: title(제목), overview(줄거리)
data[['title', 'overview']].head()

,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


#### 2) 데이터 정제하기

In [18]:
# 결측치 확인
print('overview 열의 결측치 수 :', data['overview'].isnull().sum())
print('중복된 제목 수 :', data['title'].duplicated().sum())

overview 열의 결측치 수 : 3
중복된 제목 수 : 3


In [19]:
data['overview'] = data['overview'].fillna('')                            # 줄거리가 없는 영화는 빈 문자열로
data = data.drop_duplicates(subset=['title']).reset_index(drop=True)      # 제목이 같은 영화는 하나만 남김
print('정제 후 영화 개수 :', len(data))

정제 후 영화 개수 : 4800


#### 3) 토큰화와 불용어 제거

* 영어 줄거리이므로 소문자로 바꾼 뒤 정규 표현식으로 알파벳 덩어리만 추출
* 불용어는 영어에서 자주 쓰이는 단어들을 직접 정의 (10-2의 한국어 불용어와 같은 방식)

In [20]:
stopwords = ['the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'at', 'for', 'with', 'from', 'by', 'about', 'as', 'into',
             'is', 'are', 'was', 'were', 'be', 'been', 'has', 'have', 'had', 'do', 'does', 'did', 'will', 'can', 'must',
             'he', 'she', 'it', 'they', 'his', 'her', 'its', 'their', 'him', 'them', 'who', 'that', 'this', 'which', 'what',
             'when', 'where', 'while', 'after', 'before', 'up', 'out', 'one', 'all', 'not', 'no', 's', 't']

In [21]:
docs = []
for overview in data['overview']:
    tokens = re.findall('[a-z]+', overview.lower())                  # 토큰화
    docs.append([word for word in tokens if word not in stopwords])  # 불용어 제거

print(data['title'][0], ':', docs[0])

Avatar : ['nd', 'century', 'paraplegic', 'marine', 'dispatched', 'moon', 'pandora', 'unique', 'mission', 'becomes', 'torn', 'between', 'following', 'orders', 'protecting', 'alien', 'civilization']


#### 4) 단어 집합 만들기

* 10-2에서 등장 빈도가 3회 미만인 단어를 제외했던 것처럼, 여기서는 **한 영화에만 등장하는 단어**를 제외
    - 다른 영화와 겹칠 수 없는 단어는 유사도 계산에 아무 도움이 되지 않으면서 행렬 크기만 키움
* 단어별 "등장한 문서 수"(df)를 세기 위해, 각 문서 안의 중복은 `set()`으로 제거한 뒤 `Counter`로 셈

In [22]:
df_counter = Counter()
for tokens in docs:
    df_counter.update(set(tokens))   # 한 영화 안에서 여러 번 나와도 1번만 셈 → 등장한 영화 수

print('전체 단어 수 :', len(df_counter))
print("단어 'batman'이 등장한 영화 수 :", df_counter['batman'])
print("단어 'love'가 등장한 영화 수 :", df_counter['love'])

전체 단어 수 : 20863
단어 'batman'이 등장한 영화 수 : 8
단어 'love'가 등장한 영화 수 : 379


In [23]:
vocab = sorted([word for word, df in df_counter.items() if df >= 2]) # 2편 이상의 영화에 등장하는 단어만
word_to_index = {word: index for index, word in enumerate(vocab)}
print('사용할 단어 수 :', len(vocab))

사용할 단어 수 : 10746


#### 5) TF-IDF 행렬 만들기

* 1절의 `make_bow`와 같은 일을 하지만, 영화가 4,800편이고 단어가 1만 개가 넘으므로 numpy 배열에 바로 채움
* 메모리를 아끼기 위해 `float32`를 사용 (4,800 × 10,746 × 4바이트 ≈ 200MB)

In [24]:
N, V = len(docs), len(vocab)
tf = np.zeros((N, V), dtype=np.float32)

for i, tokens in enumerate(docs):
    for word in tokens:
        if word in word_to_index:                # 단어 집합에 있는 단어만
            tf[i, word_to_index[word]] += 1

print('DTM(tf)의 크기 :', tf.shape)   # (영화 수, 단어 수)

DTM(tf)의 크기 : (4800, 10746)


In [25]:
df = (tf > 0).sum(axis=0)         # 각 단어가 등장한 영화 수
idf = np.log(N / (1 + df))
tfidf = tf * idf

print('TF-IDF 행렬의 크기 :', tfidf.shape)

TF-IDF 행렬의 크기 : (4800, 10746)


* 영화 한 편이 약 1만 차원의 벡터로 표현됨 (단어 집합의 크기 = 열의 수)

#### 6) 코사인 유사도 구하기

In [26]:
cosine_sim = cos_sim_matrix(tfidf)
print('코사인 유사도 행렬의 크기 :', cosine_sim.shape)

코사인 유사도 행렬의 크기 : (4800, 4800)


* `cosine_sim[i][j]`: $i$번째 영화와 $j$번째 영화 줄거리의 유사도
* 영화 제목으로 인덱스를 찾을 수 있도록 딕셔너리를 만들어 둠

In [27]:
title_to_index = dict(zip(data['title'], data.index))
print(title_to_index['The Dark Knight Rises'])

3


In [28]:
idx = title_to_index['The Dark Knight Rises']
print(cosine_sim[idx][:10].round(3))  # 다크 나이트 라이즈와 처음 10개 영화의 유사도

[0.024 0.    0.    1.    0.01  0.003 0.01  0.029 0.019 0.143]


#### 7) 추천 함수 만들기

In [29]:
def get_recommendations(title, n=10):
    idx = title_to_index[title]                                     # 1. 입력 영화의 인덱스
    sim_scores = list(enumerate(cosine_sim[idx]))                   # 2. (영화 인덱스, 유사도) 쌍의 리스트
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) # 3. 유사도가 높은 순으로 정렬
    sim_scores = sim_scores[1:n+1]                                  # 4. 0번은 자기 자신이므로 제외하고 n개
    movie_indices = [i for i, score in sim_scores]                  # 5. 영화 인덱스만 추출

    return pd.DataFrame({'title': data['title'].iloc[movie_indices].values,
                         'similarity': [round(float(score), 3) for i, score in sim_scores]})

In [30]:
get_recommendations('The Dark Knight Rises')

,title,similarity
0,Batman Forever,0.324
1,Batman Returns,0.307
2,The Dark Knight,0.304
3,Batman,0.269
4,"Batman: The Dark Knight Returns, Part 2",0.179
5,Slow Burn,0.179
6,Batman Begins,0.160
7,Batman v Superman: Dawn of Justice,0.143
8,Batman & Robin,0.134
9,JFK,0.133


In [31]:
get_recommendations('Toy Story')

,title,similarity
0,Toy Story 3,0.505
1,Toy Story 2,0.443
2,The 40 Year Old Virgin,0.303
3,Man on the Moon,0.178
4,Factory Girl,0.171
5,Class of 1984,0.169
6,For Your Consideration,0.162
7,Heartbeeps,0.150
8,Losin' It,0.128
9,The Man,0.120


In [32]:
get_recommendations('Spider-Man', n=5)

,title,similarity
0,Spider-Man 3,0.291
1,Spider-Man 2,0.239
2,The Amazing Spider-Man 2,0.230
3,The Amazing Spider-Man,0.228
4,Arachnophobia,0.191


* 줄거리에 공통으로 등장하는 단어가 많을수록 유사도가 높음
* 어떤 단어 때문에 비슷하다고 판단했는지 확인해 보기: 두 영화의 TF-IDF 값을 곱해서 큰 순서대로 출력

In [33]:
def common_words(title1, title2, n=10):
    v1 = tfidf[title_to_index[title1]]
    v2 = tfidf[title_to_index[title2]]
    product = v1 * v2                               # 두 벡터를 원소별로 곱함 (둘 다 0이 아닌 단어만 남음)
    top = product.argsort()[::-1][:n]               # 값이 큰 순서대로 n개
    return [vocab[i] for i in top if product[i] > 0]

print(common_words('The Dark Knight Rises', 'The Dark Knight'))
print(common_words('Toy Story', 'Toy Story 2'))
print(common_words('Spider-Man', 'Arachnophobia'))

['batman', 'dent', 'gotham', 'attorney', 'harvey', 'district']
['woody', 'andy', 'buzz', 'toys', 'lightyear', 'owner', 'heart']
['spider']


### 5. 정리

* BoW, TF-IDF: 단어의 **빈도**로 문서를 벡터로 표현
* 코사인 유사도: 두 벡터의 **방향**이 얼마나 비슷한지로 문서의 유사도를 계산
* 딥러닝 없이도 쓸 만한 추천 시스템을 만들 수 있음
* 한계: 단어의 **철자가 같아야** 같은 단어로 인식
    - "spider"라는 단어 때문에 스파이더맨과 거미 공포 영화가 비슷하다고 판단
    - "car"와 "automobile", "무섭다"와 "공포스럽다"는 전혀 다른 단어로 취급됨
    - 단어의 **의미**가 비슷한 것까지 잡아내려면 10-2에서 배운 워드 임베딩이 필요함

### 6. 연습문제

1. 자신이 좋아하는 영화로 추천을 받아 보세요. 영화 제목이 정확히 기억나지 않으면 아래 코드로 검색할 수 있습니다.

In [34]:
# 제목에 'Harry'가 들어가는 영화 검색
data[data['title'].str.contains('Harry')]['title']

8          Harry Potter and the Half-Blood Prince
113     Harry Potter and the Order of the Phoenix
114           Harry Potter and the Goblet of Fire
191      Harry Potter and the Prisoner of Azkaban
197      Harry Potter and the Philosopher's Stone
276       Harry Potter and the Chamber of Secrets
1621      Dumb and Dumberer: When Harry Met Lloyd
2213                         Deconstructing Harry
2455                      When Harry Met Sally...
3331                                  Harry Brown
4294                       The Trouble with Harry
Name: title, dtype: object

In [ ]:
# 여기에 코드를 작성하세요
get_recommendations('')

2. TF-IDF 대신 단순 빈도(`tf`)로 코사인 유사도를 구해서 같은 영화를 추천받아 보세요. 결과가 어떻게 달라지나요?

In [ ]:
# 여기에 코드를 작성하세요
cosine_sim_tfidf = cosine_sim   # TF-IDF로 구한 결과는 따로 저장해 둠
cosine_sim = None               # 적절한 코드를 작성하세요
get_recommendations('')